In [ ]:
import os
import yfinance as yf
import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive
drive.mount('/content/drive')
print("🚀 啟動 Yahoo 全市場日線挖掘機...")

# 1. 設定新資料夾 (專門放日線)
DAILY_DIR = '/content/drive/MyDrive/MarketMamba_Database/Macro_1D_History'
os.makedirs(DAILY_DIR, exist_ok=True)

# 2. 讀取你的股票名單
TRACKER_PATH = '/content/drive/MyDrive/MarketMamba_Database/shioaji_tracker.csv'
df_tracker = pd.read_csv(TRACKER_PATH, dtype={'Ticker': str})
stock_list = df_tracker['Ticker'].dropna().tolist()

print(f"📊 準備抓取 {len(stock_list)} 檔股票的歷史日線...")

success_count = 0
for ticker in tqdm(stock_list, desc="日線下載中"):
    # Yahoo Finance 的台股代號需要加上 .TW (上市) 或 .TWO (上櫃)
    # 我們先無腦加 .TW 試試看，抓不到再試 .TWO
    yf_ticker = f"{ticker}.TW"

    try:
        # period="max" 直接要歷史上所有的日線！
        df_daily = yf.download(yf_ticker, period="max", progress=False)

        # 如果 .TW 抓不到，代表它可能是上櫃公司，改用 .TWO
        if df_daily.empty:
            yf_ticker = f"{ticker}.TWO"
            df_daily = yf.download(yf_ticker, period="max", progress=False)

        if not df_daily.empty:
            # 清理一下欄位結構 (有時候 yf 會回傳 MultiIndex)
            if isinstance(df_daily.columns, pd.MultiIndex):
                df_daily.columns = df_daily.columns.droplevel(1)

            # 存成 Parquet
            file_path = os.path.join(DAILY_DIR, f"{ticker}_1D.parquet")
            df_daily.to_parquet(file_path)
            success_count += 1

    except Exception as e:
        pass

print(f"\n✅ 宏觀日線大本營建立完成！成功抓取 {success_count} 檔股票的終極歷史日線！")

In [ ]:
import os
import pandas as pd
from tqdm.auto import tqdm

print("🏥 啟動宏觀日線資料庫體檢中心...\n")

DAILY_DIR = '/content/drive/MyDrive/MarketMamba_Database/Macro_1D_History'

try:
    files = [f for f in os.listdir(DAILY_DIR) if f.endswith('_1D.parquet')]

    if not files:
        print("❌ 找不到任何日線檔案，請確認 Yahoo 挖掘機是不是存錯地方了！")
    else:
        print(f"📁 總共尋獲 {len(files)} 個日線檔案，開始逐一掃描...")

        healthy_count = 0
        empty_files = []
        short_files = []
        missing_cols = []

        for f in tqdm(files, desc="日線體檢中"):
            ticker = f.split('_')[0]
            file_path = os.path.join(DAILY_DIR, f)

            try:
                df = pd.read_parquet(file_path)

                # 1. 檢查是否為空
                if df.empty:
                    empty_files.append(ticker)
                    # 順手把空檔案刪掉，免得害死後面的 GNN
                    os.remove(file_path)
                    continue

                # 2. 檢查關鍵欄位 (算相關係數一定要有收盤價)
                if 'Close' not in df.columns:
                    missing_cols.append(ticker)
                    os.remove(file_path)
                    continue

                # 3. 檢查資料長度 (上市不到半年的新股，算相關係數沒意義)
                if len(df) < 120:  # 大約半年
                    short_files.append(f"{ticker} (僅 {len(df)} 筆)")
                    os.remove(file_path)
                    continue

                healthy_count += 1

            except Exception as e:
                empty_files.append(f"{ticker} (讀取失敗)")
                os.remove(file_path) # 壞掉的檔案也刪掉

        print("\n🏆 【宏觀日線體檢報告】 🏆")
        print(f"✅ 健康合格，獲准進入 GNN 鍛造爐: {healthy_count} 檔")

        if empty_files:
            print(f"🚨 空檔案或損壞 (已自動清除): {len(empty_files)} 檔 (例如: {empty_files[:5]})")
        if missing_cols:
            print(f"🚨 缺少 Close 欄位 (已自動清除): {len(missing_cols)} 檔")
        if short_files:
            print(f"⚠️ 歷史太短缺乏連動參考價值 (已自動清除): {len(short_files)} 檔 (例如: {short_files[:5]})")

        print("\n✨ 清理完畢！現在資料夾裡剩下的都是純度 100% 的黃金日線了！")

except Exception as e:
    print(f"❌ 發生未預期錯誤: {e}")

In [ ]:
import os
import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive
drive.mount('/content/drive')

print("📅 啟動宏觀日線時間軸掃描器...\n")

DAILY_DIR = '/content/drive/MyDrive/MarketMamba_Database/Macro_1D_History'
files = [f for f in os.listdir(DAILY_DIR) if f.endswith('_1D.parquet')]

if not files:
    print("❌ 找不到檔案！請確認路徑是否正確。")
else:
    print(f"📁 準備掃描 {len(files)} 檔股票的歷史區間...")

    stats = []

    for f in tqdm(files, desc="掃描時間軸"):
        ticker = f.split('_')[0]
        file_path = os.path.join(DAILY_DIR, f)

        try:
            df = pd.read_parquet(file_path)

            # 確保 index 是時間格式
            if not pd.api.types.is_datetime64_any_dtype(df.index):
                df.index = pd.to_datetime(df.index)

            start_date = df.index.min()
            end_date = df.index.max()

            stats.append({
                'Ticker': ticker,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Total_Days': len(df)
            })
        except Exception as e:
            pass

    df_stats = pd.DataFrame(stats)

    # 將時間轉為純日期格式方便閱讀
    df_stats['Start_Date'] = df_stats['Start_Date'].dt.date
    df_stats['End_Date'] = df_stats['End_Date'].dt.date

    print("\n==========================================")
    print("📈 【宏觀日線時間軸報告】 📈")
    print("==========================================")
    print(f"🔹 全市場最早資料起點: {df_stats['Start_Date'].min()}")
    print(f"🔹 全市場最晚資料終點: {df_stats['End_Date'].max()}")
    print(f"🔹 平均每檔股票擁有天數: {int(df_stats['Total_Days'].mean())} 天 (約 {int(df_stats['Total_Days'].mean()/252)} 年)")

    print("\n⏳ 歷史最悠久的台股元老 (前 5 名):")
    print(df_stats.sort_values('Start_Date').head(5).to_string(index=False))

    print("\n⏱️ 較晚掛牌的年輕公司 (前 5 名):")
    # 排除昨天被我們踢掉的，看看剩下的裡面最年輕的是誰
    print(df_stats.sort_values('Start_Date', ascending=False).head(5).to_string(index=False))

    # 檢查有沒有人的終點不是最近的日期 (抓出可能提早下市或停止交易的股票)
    latest_market_date = df_stats['End_Date'].max()
    stale_stocks = df_stats[df_stats['End_Date'] < latest_market_date]
    if not stale_stocks.empty:
        print(f"\n⚠️ 發現 {len(stale_stocks)} 檔股票的資料終點沒有跟上大盤 (可能已下市或暫停交易):")
        print(stale_stocks.sort_values('End_Date').head(5).to_string(index=False))